In [ ]:
import pandas as pd
df=pd.DataFrame({'dob':['1995-03-15','1988-11-02','2001-07-20']})
df['dob']=pd.to_datetime(df['dob'])
df['age']=2024-df['dob'].dt.year
df['birth_month']=df['dob'].dt.month
df['birth_day']=df['dob'].dt.day_name()
bins=[0,28,44,60,100]
labels=['Gen-Z','Millennial','Gen-X','Boomer']
df['generation']=pd.cut(df['age'],bins=bins,labels=labels)
print(df[['age','birth_month','birth_day','generation']])

   age  birth_month  birth_day  generation
0   29            3  Wednesday  Millennial
1   36           11  Wednesday  Millennial
2   23            7     Friday       Gen-Z


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
df['purchased']=['Yes','No','Yes']
df['size']=['Small','Medium','Large']
le=LabelEncoder()
df['purchased_enc']=le.fit_transform(df['purchased'])
size_order={'Small':0,'Medium':1,'Large':2}
df['size_enc']=df['size'].map(size_order)
print(df[['purchased','purchased_enc','size','size_enc']])

  purchased  purchased_enc    size  size_enc
0       Yes              1   Small         0
1        No              0  Medium         1
2       Yes              1   Large         2


In [ ]:
import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Chennai', 'Delhi']})
df_encoded = pd.get_dummies(df, columns=['city'], dtype=int)
print(df_encoded)

print("-----------------------------------------------")
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(df[['city']])

print(encoded)

   city_Chennai  city_Delhi  city_Mumbai
0             0           1            0
1             0           0            1
2             1           0            0
3             0           1            0
-----------------------------------------------
[[1. 0.]
 [0. 1.]
 [0. 0.]
 [1. 0.]]


In [ ]:
import pandas as pd
import numpy as np
df=pd.DataFrame({
    'feature_A':[1,2,np.nan,4,5],
    'feature_B':['X','Y','Z',np.nan,'W'],
    'feature_C':[10.5,np.nan,12.3,14.0,np.nan]
})
print('Missing values count per column:')
print(df.isnull().sum())
print('\n')
print('percentage missing per column:')
print(df.isnull().mean()*100)
print('\n')

Missing values count per column:
feature_A    1
feature_B    1
feature_C    2
dtype: int64


percentage missing per column:
feature_A    20.0
feature_B    20.0
feature_C    40.0
dtype: float64




In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
numeric_cols = ['feature_A', 'feature_C']
num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

categorical_cols = ['feature_B']
cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


print('Total missing values after imputation:', df.isnull().sum().sum()) # Should print 0
print('\nDataFrame after imputation:')
print(df)

Total missing values after imputation: 0

DataFrame after imputation:
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         W       14.0
4        5.0         W       12.3


In [ ]:
Q1 = df['feature_A'].quantile(0.25)
Q3 = df['feature_A'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df_clean_iqr = df[(df['feature_A'] >= lower) & (df['feature_A'] <= upper)]
print("DataFrame after IQR outlier removal on 'feature_A':")
print(df_clean_iqr)
print('\n')
from scipy import stats
z_scores = stats.zscore(df['feature_A'])
df_clean_zscore = df[abs(z_scores) < 3]
print("DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):")
print(df_clean_zscore)


DataFrame after IQR outlier removal on 'feature_A':
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         W       14.0
4        5.0         W       12.3


DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):
   feature_A feature_B  feature_C
0        1.0         X       10.5
1        2.0         Y       12.3
2        3.0         Z       12.3
3        4.0         W       14.0
4        5.0         W       12.3


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Assuming 'df' is the DataFrame from previous steps
# For demonstration, let's define X and y from the existing 'df'
X = df[['feature_A', 'feature_C']] # Using numeric features as X
# Create a dummy target variable 'y' for classification example
# e.g., if feature_A is greater than its median, classify as 1, else 0
y = (df['feature_A'] > df['feature_A'].median()).astype(int)

# Step 1: split data BEFORE building the pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 2: define the pipeline (a list of (name, transformer) tuples)
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # fill NaNs
    ('scaler', StandardScaler()), # scale features
    ('model', LogisticRegression(random_state=42)), # train classifier
])

# Step 3: fit ONLY on training data — one call does everything
pipe.fit(X_train, y_train)

# Step 4: evaluate — the pipeline automatically transforms X_test
# using parameters learned from X_train (no leakage!)
print('Accuracy:', pipe.score(X_test, y_test))

# Predict new data in production — just one call
# For demonstration, create a sample X_new based on df's columns
X_new = pd.DataFrame({
    'feature_A': [6.0, np.nan, 8.0],
    'feature_C': [15.0, 16.0, np.nan]
})
predictions = pipe.predict(X_new)
print('Predictions for new data:', predictions)

Accuracy: 1.0
Predictions for new data: [1 1 1]
